# 03 — Segmentation de l'eau avec U-Net et encodeur préentraîné

L'objectif est de prédire un masque binaire eau / non-eau aussi proche que possible du masque réel.

Cette version améliore la baseline précédente sur quatre points :

- encodeur **ResNet18 préentraîné sur ImageNet** au lieu d'un encodeur appris depuis zéro ;
- images de **512 × 512** avec conservation du ratio et padding ;
- interpolation bilinéaire pour l'image et **nearest-neighbor pour le masque** ;
- batch 1 compatible avec 4 Go de VRAM, mixed precision et accumulation de gradients.

Le préentraînement ne fournit pas directement un détecteur d'eau. Il fournit des filtres visuels génériques déjà capables de reconnaître contours, textures et formes. Ces poids sont ensuite ajustés sur Water-v2 avec un learning rate plus faible que celui du décodeur. Au premier lancement, PyTorch télécharge automatiquement les poids officiels ResNet18 si ceux-ci ne sont pas déjà en cache.


In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.append(str(ROOT / 'src'))

from water_detection_methods.data import (
    find_water_v2_pairs,
    load_pair_with_padding,
)
from water_detection_methods.visualization import overlay_mask, plot_metric_bars


## Configuration CUDA et mémoire

La configuration par défaut est volontairement conservatrice pour une GTX 1650 Max-Q ou une RTX 3050 Laptop avec 4 Go de VRAM :

- résolution : 512 × 512 ;
- batch physique : 1 ;
- accumulation : 8 étapes, soit un batch effectif de 8 ;
- mixed precision FP16 ;
- BatchNorm de l'encodeur préentraîné figée, car un batch de 1 donne des statistiques instables.

Si la RTX 3050 accepte un batch de 2, `BATCH_SIZE` peut être augmenté et `GRAD_ACCUMULATION_STEPS` réduit à 4 afin de conserver le même batch effectif.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA est indisponible. Sélectionner le kernel CUDA du projet avant de continuer.'
    )

DEVICE = torch.device('cuda')
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

IMAGE_SIZE = (512, 512)  # Convention Pillow : (largeur, hauteur).
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
NUM_WORKERS = 0  # Configuration robuste pour Jupyter sous Windows.
EPOCHS = 60
ENCODER_LEARNING_RATE = 1e-4
DECODER_LEARNING_RATE = 3e-4
PATIENCE = 10
THRESHOLD = 0.5
USE_AMP = True
USE_PRETRAINED_WEIGHTS = True

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

properties = torch.cuda.get_device_properties(0)
print('PyTorch             :', torch.__version__)
print('Runtime CUDA PyTorch:', torch.version.cuda)
print('GPU                 :', torch.cuda.get_device_name(0))
print('Mémoire GPU         :', f'{properties.total_memory / 1024**3:.1f} Go')
print('Batch effectif      :', BATCH_SIZE * GRAD_ACCUMULATION_STEPS)


## Séparation du dataset Water-v2

La séparation reste effectuée par groupe afin d'empêcher que des images voisines d'une même source apparaissent à la fois pendant l'entraînement et l'évaluation. Le split actuel constitue un benchmark de généralisation : les groupes d'évaluation sont différents des deux groupes d'entraînement.


In [ ]:
DATASET_DIR = ROOT / 'water_v2'
IMAGES_DIR = DATASET_DIR / 'JPEGImages'

pairs = find_water_v2_pairs(DATASET_DIR)
if not pairs:
    raise FileNotFoundError(f'Aucune paire image/masque trouvée dans {DATASET_DIR}')

def lire_groupes(path):
    return {
        line.strip()
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    }

def groupe_de_la_paire(pair):
    image_path, _ = pair
    return image_path.relative_to(IMAGES_DIR).parts[0]

train_groups = lire_groupes(DATASET_DIR / 'train.txt')
evaluation_groups = lire_groupes(DATASET_DIR / 'val.txt')
train_pairs = [pair for pair in pairs if groupe_de_la_paire(pair) in train_groups]
evaluation_pairs = [
    pair for pair in pairs if groupe_de_la_paire(pair) in evaluation_groups
]

covered_groups = train_groups | evaluation_groups
unused_pairs = [pair for pair in pairs if groupe_de_la_paire(pair) not in covered_groups]
if unused_pairs:
    raise ValueError(f'{len(unused_pairs)} paires appartiennent à des groupes non déclarés.')

splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
evaluation_labels = [groupe_de_la_paire(pair) for pair in evaluation_pairs]
val_indices, test_indices = next(
    splitter.split(evaluation_pairs, groups=evaluation_labels)
)
val_pairs = [evaluation_pairs[index] for index in val_indices]
test_pairs = [evaluation_pairs[index] for index in test_indices]

train_group_names = {groupe_de_la_paire(pair) for pair in train_pairs}
val_group_names = {groupe_de_la_paire(pair) for pair in val_pairs}
test_group_names = {groupe_de_la_paire(pair) for pair in test_pairs}
assert train_group_names.isdisjoint(val_group_names)
assert train_group_names.isdisjoint(test_group_names)
assert val_group_names.isdisjoint(test_group_names)

print(f'Paires totales : {len(pairs)}')
print(f'Train          : {len(train_pairs)} images, {len(train_group_names)} groupes')
print(f'Validation     : {len(val_pairs)} images, {len(val_group_names)} groupes')
print(f'Test           : {len(test_pairs)} images, {len(test_group_names)} groupes')


## Pipeline 512 × 512 avec conservation du ratio

Le letterboxing calcule un facteur d'échelle unique, redimensionne l'image sans la déformer, puis complète les dimensions restantes avec du padding. Le même padding est appliqué au masque.

L'image utilise une interpolation bilinéaire. Le masque utilise obligatoirement nearest-neighbor afin de conserver uniquement les valeurs de classe 0 et 1. Les images sont ensuite normalisées avec la moyenne et l'écart-type ImageNet attendus par ResNet18.


In [ ]:
class WaterSegmentationDataset(Dataset):
    def __init__(self, pairs, image_size=IMAGE_SIZE, augment=False):
        self.pairs = list(pairs)
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image, mask = load_pair_with_padding(
            image_path,
            mask_path,
            size=self.image_size,
        )

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.3:
                brightness = random.uniform(0.85, 1.15)
                image = np.clip(image * brightness, 0.0, 1.0)

        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = np.ascontiguousarray(image.transpose(2, 0, 1), dtype=np.float32)
        mask = np.ascontiguousarray(mask[None, ...], dtype=np.float32)
        return torch.from_numpy(image), torch.from_numpy(mask)

def denormaliser_image(image_tensor):
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(image * IMAGENET_STD + IMAGENET_MEAN, 0.0, 1.0)

train_dataset = WaterSegmentationDataset(train_pairs, augment=True)
train_eval_dataset = WaterSegmentationDataset(train_pairs, augment=False)
val_dataset = WaterSegmentationDataset(val_pairs, augment=False)
test_dataset = WaterSegmentationDataset(test_pairs, augment=False)

loader_options = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': True,
}
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, shuffle=True, drop_last=False,
    generator=loader_generator, **loader_options
)
train_eval_loader = DataLoader(train_eval_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

images_batch, masks_batch = next(iter(train_loader))
print('Batch images :', images_batch.shape, images_batch.dtype)
print('Batch masques:', masks_batch.shape, masks_batch.dtype)
print('Valeurs masque:', torch.unique(masks_batch).tolist())

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
axes[0].imshow(denormaliser_image(images_batch[0]))
axes[0].set_title('Image 512 × 512 avec padding')
axes[1].imshow(masks_batch[0, 0], cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Masque nearest-neighbor')
for axis in axes:
    axis.axis('off')
plt.show()


## U-Net avec encodeur ResNet18 préentraîné

ResNet18 joue le rôle d'encodeur. Ses poids ImageNet donnent un point de départ visuel robuste. Le décodeur U-Net, initialisé aléatoirement, reconstruit progressivement le masque à pleine résolution à partir des représentations de l'encodeur et des skip connections.

Le décodeur utilise GroupNorm, plus stable que BatchNorm avec un batch physique de 1. Les BatchNorm de ResNet18 restent en mode évaluation pendant le fine-tuning.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        groups = min(8, out_channels)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        return self.conv(torch.cat((x, skip), dim=1))

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride,
            padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + identity)

class ResNet18Encoder(nn.Module):
    WEIGHTS_URL = 'https://download.pytorch.org/models/resnet18-f37072fd.pth'

    def __init__(self, pretrained=True):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(
            3, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=2)

        if pretrained:
            state_dict = torch.hub.load_state_dict_from_url(
                self.WEIGHTS_URL, map_location='cpu', progress=True
            )
            incompatible = self.load_state_dict(state_dict, strict=False)
            expected_unused = {'fc.weight', 'fc.bias'}
            if set(incompatible.unexpected_keys) != expected_unused:
                raise RuntimeError(
                    f'Clés ResNet18 inattendues : {incompatible.unexpected_keys}'
                )
            if incompatible.missing_keys:
                raise RuntimeError(
                    f'Clés ResNet18 manquantes : {incompatible.missing_keys}'
                )

    def _make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, out_channels, stride=stride)]
        self.in_channels = out_channels
        layers.extend(
            BasicBlock(self.in_channels, out_channels) for _ in range(1, blocks)
        )
        return nn.Sequential(*layers)

class PretrainedResNet18UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.encoder = ResNet18Encoder(pretrained=pretrained)

        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 64)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            DoubleConv(32, 32),
        )
        self.output = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        skip0 = self.encoder.relu(self.encoder.bn1(self.encoder.conv1(x)))
        skip1 = self.encoder.layer1(self.encoder.maxpool(skip0))
        skip2 = self.encoder.layer2(skip1)
        skip3 = self.encoder.layer3(skip2)
        x = self.encoder.layer4(skip3)

        x = self.up4(x, skip3)
        x = self.up3(x, skip2)
        x = self.up2(x, skip1)
        x = self.up1(x, skip0)
        x = self.up0(x)
        return self.output(x)

    def freeze_encoder_batch_norm(self):
        for module in self.encoder.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

model = PretrainedResNet18UNet(
    pretrained=USE_PRETRAINED_WEIGHTS
).to(DEVICE)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(f'Paramètres totaux     : {total_parameters:,}')
print(f'Paramètres entraînables: {trainable_parameters:,}')


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probabilities = torch.sigmoid(logits.float())
        dimensions = (1, 2, 3)
        intersection = (probabilities * targets).sum(dim=dimensions)
        denominator = probabilities.sum(dim=dimensions) + targets.sum(dim=dimensions)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        return 1.0 - dice.mean()

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def segmentation_loss(logits, targets):
    return 0.5 * bce_loss(logits, targets) + 0.5 * dice_loss(logits, targets)

@torch.no_grad()
def batch_metrics(logits, targets, threshold=THRESHOLD):
    predictions = torch.sigmoid(logits.float()) >= threshold
    targets_binary = targets >= 0.5
    dimensions = (1, 2, 3)

    tp = (predictions & targets_binary).sum(dim=dimensions).float()
    fp = (predictions & ~targets_binary).sum(dim=dimensions).float()
    fn = (~predictions & targets_binary).sum(dim=dimensions).float()
    union = tp + fp + fn

    dice = (2.0 * tp + 1.0) / (2.0 * tp + fp + fn + 1.0)
    iou = (tp + 1.0) / (union + 1.0)
    precision = (tp + 1.0) / (tp + fp + 1.0)
    recall = (tp + 1.0) / (tp + fn + 1.0)
    accuracy = (predictions == targets_binary).float().mean(dim=dimensions)

    return {
        'dice': dice.mean().item(),
        'iou': iou.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
        'accuracy': accuracy.mean().item(),
    }

encoder_parameter_ids = {id(parameter) for parameter in model.encoder.parameters()}
decoder_parameters = [
    parameter for parameter in model.parameters()
    if id(parameter) not in encoder_parameter_ids
]
optimizer = torch.optim.AdamW(
    [
        {'params': model.encoder.parameters(), 'lr': ENCODER_LEARNING_RATE},
        {'params': decoder_parameters, 'lr': DECODER_LEARNING_RATE},
    ],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

def run_epoch(loader, training):
    model.train(training)
    if training:
        model.freeze_encoder_batch_norm()
        optimizer.zero_grad(set_to_none=True)

    metric_names = ('dice', 'iou', 'precision', 'recall', 'accuracy')
    totals = {'loss': 0.0, **{name: 0.0 for name in metric_names}}
    sample_count = 0

    for batch_index, (images, masks) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        batch_size = images.size(0)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(
                device_type='cuda', dtype=torch.float16, enabled=USE_AMP
            ):
                logits = model(images)
                loss = segmentation_loss(logits, masks)

            if training:
                scaler.scale(loss / GRAD_ACCUMULATION_STEPS).backward()
                should_step = (
                    (batch_index + 1) % GRAD_ACCUMULATION_STEPS == 0
                    or batch_index + 1 == len(loader)
                )
                if should_step:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        metrics = batch_metrics(logits.detach(), masks)
        totals['loss'] += loss.item() * batch_size
        for name in metric_names:
            totals[name] += metrics[name] * batch_size
        sample_count += batch_size

    return {name: value / sample_count for name, value in totals.items()}


In [ ]:
CHECKPOINT_PATH = ROOT / 'models' / 'unet_resnet18_512_best.pt'
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'train_iou': [], 'val_iou': [],
}
best_val_iou = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics['iou'])

    for metric in ('loss', 'dice', 'iou'):
        history[f'train_{metric}'].append(train_metrics[metric])
        history[f'val_{metric}'].append(val_metrics[metric])

    encoder_lr = optimizer.param_groups[0]['lr']
    decoder_lr = optimizer.param_groups[1]['lr']
    print(
        f'Epoch {epoch:03d}/{EPOCHS} | '
        f'loss {train_metrics["loss"]:.4f}/{val_metrics["loss"]:.4f} | '
        f'Dice {train_metrics["dice"]:.4f}/{val_metrics["dice"]:.4f} | '
        f'IoU {train_metrics["iou"]:.4f}/{val_metrics["iou"]:.4f} | '
        f'lr enc/dec {encoder_lr:.1e}/{decoder_lr:.1e}'
    )

    if val_metrics['iou'] > best_val_iou:
        best_val_iou = val_metrics['iou']
        epochs_without_improvement = 0
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_metrics': train_metrics,
                'val_metrics': val_metrics,
                'config': {
                    'image_size': IMAGE_SIZE,
                    'batch_size': BATCH_SIZE,
                    'gradient_accumulation_steps': GRAD_ACCUMULATION_STEPS,
                    'threshold': THRESHOLD,
                    'pretrained_encoder': USE_PRETRAINED_WEIGHTS,
                },
            },
            CHECKPOINT_PATH,
        )
        print(f'  Meilleur modèle sauvegardé : IoU={best_val_iou:.4f}')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping après {epoch} epochs.')
            break

print('Checkpoint :', CHECKPOINT_PATH)


In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

for axis, metric, title in zip(
    axes, ('loss', 'dice', 'iou'), ('Perte', 'Dice', 'IoU')
):
    axis.plot(epochs_ran, history[f'train_{metric}'], label='Train')
    axis.plot(epochs_ran, history[f'val_{metric}'], label='Validation')
    axis.set_title(title)
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

train_eval_metrics = run_epoch(train_eval_loader, training=False)
val_eval_metrics = run_epoch(val_loader, training=False)
test_metrics = run_epoch(test_loader, training=False)

print(f"Checkpoint de l'epoch : {checkpoint['epoch']}")
for split_name, metrics in {
    'Train': train_eval_metrics,
    'Validation': val_eval_metrics,
    'Test': test_metrics,
}.items():
    print(
        f'{split_name:10s} | accuracy={metrics["accuracy"]:.4f} | '
        f'precision={metrics["precision"]:.4f} | recall={metrics["recall"]:.4f} | '
        f'Dice={metrics["dice"]:.4f} | IoU={metrics["iou"]:.4f}'
    )

resultats_par_split = {
    'Train': train_eval_metrics,
    'Validation': val_eval_metrics,
    'Test': test_metrics,
}
plot_metric_bars(resultats_par_split)
plt.show()


In [ ]:
@torch.inference_mode()
def visualiser_predictions(model, pairs, n_images=8, seed=SEED, threshold=THRESHOLD):
    if not pairs:
        raise ValueError('Aucune paire image/masque à visualiser.')

    rng = np.random.default_rng(seed)
    selected = rng.choice(len(pairs), size=min(n_images, len(pairs)), replace=False)
    fig, axes = plt.subplots(
        len(selected), 4, figsize=(16, 3 * len(selected)),
        squeeze=False, constrained_layout=True
    )
    titles = ('Image', 'Masque réel', 'Masque prédit', 'Overlay')
    for column, title in enumerate(titles):
        axes[0, column].set_title(title)

    model.eval()
    for row, pair_index in enumerate(selected):
        image_path, mask_path = pairs[pair_index]
        image, true_mask = load_pair_with_padding(
            image_path, mask_path, size=IMAGE_SIZE
        )
        normalized = (image - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(
            np.ascontiguousarray(normalized.transpose(2, 0, 1), dtype=np.float32)
        ).unsqueeze(0).to(DEVICE)

        with torch.amp.autocast(
            device_type='cuda', dtype=torch.float16, enabled=USE_AMP
        ):
            probability = torch.sigmoid(model(tensor).float())[0, 0]
        pred_mask = (probability >= threshold).byte().cpu().numpy()

        axes[row, 0].imshow(image)
        axes[row, 1].imshow(true_mask, cmap='Blues', vmin=0, vmax=1)
        axes[row, 2].imshow(pred_mask, cmap='Blues', vmin=0, vmax=1)
        axes[row, 3].imshow(overlay_mask(image, pred_mask))
        axes[row, 0].set_ylabel(groupe_de_la_paire((image_path, mask_path)))
        for axis in axes[row]:
            axis.axis('off')

    fig.suptitle(f'Prédictions U-Net ResNet18 sur {len(selected)} images de test')
    plt.show()

visualiser_predictions(model, test_pairs, n_images=8)


## Étapes suivantes

1. Exécuter toutes les cellules et relever la VRAM maximale.
2. Comparer IoU, Dice, précision et rappel avec l'ancien U-Net 256 × 256.
3. Rechercher le meilleur seuil sur la validation au lieu de supposer 0,5 optimal.
4. Ajouter Boundary F1 et des métriques par groupe.
5. Utiliser la fonction `plot_metric_bars` pour comparer ensuite U-Net, U-Net++, DeepLabV3+ et les autres modèles sous le même protocole.
